#What is Transfer learning

**Transfer Learning** allows us to take patterns (also called weights) another model has learned from another problem and use them for our own problem.

Or we could take the patterns from a language model (a model that's been through large amounts of text to learn a representation of language) and use them as the basis of a model to classify different text samples.



##Why use Transfer learning

The two main reason to using Transfer Learning

1. Can leverage an existing model (usually a neural network architecture) proven to work on problems similar to our own

2. Can leverage a working model which has **already learned** patterns on similar data to our own .  This often results in achieving great results with less custom data.

###Getting the data

Download/import the required modules for this section


In [ ]:


# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install -U torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu113
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")



In [ ]:
# The regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
  from torchinfo import summary
except:
  print("[INFO] Couldn't find torchinfo ... Installing It.")
  !pip install -q torchinfo
  from torchinfo import summary

# Try import the going_modular directory, download it from the Github if it doesn't work
try:
  from going_modular.going_modular import data_set_up , engine
except:
  # Get the  going_modular scripts
  print("[INFO] Couldn't find going_modular scripts ... downloading them from Github.")
  !git clone https://github.com/mrdbourke/pytorch-deep-learning
  !mv pytorch-deep-learning/going_modular .
  !rm -rf pytorch-deep-learning
  from going_modular.going_modular import data_setup, engine


In [ ]:
# Set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

##Get the data


Before we can start to use **transfer learning**, we'll need a dataset

In [ ]:
import os
import zipfile

from pathlib import Path

data_path = Path("./")
zip_path = data_path / "pizza_steak_sushi.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
  zip_ref.extractall(data_path)


In [ ]:
# Setup the Dirs
train_dir = data_path / "train"
test_dir = data_path / "test"

# Create Datasets and DataLoaders

## Creating a transform for torchvision.models (manual creation)

In [ ]:
# Create a transforms pipeline manually (required for torchvision < 0.13)
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # 1.Reshape all images to 224x224 (though some models may require different sizes)
    transforms.ToTensor(), # 2. Turn image between zero and one.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # 3. A mean of [0.485, 0.456, 0.406] (across each colour channel)
                         std=[0.229, 0.224, 0.225])  # 4. A standard deviation of [0.229, 0.224, 0.225] (across each colour channel),
])

In [ ]:
# Create training and testing DataLoaders as well as get a list of class names
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=manual_transforms, # resize, convert images to between 0 & 1 and normalize them
                                                                               batch_size=32) # set mini-batch size to 32

train_dataloader, test_dataloader, class_names

## Creatin a transform for `torchvision.models` (auto generation)

As previously stated, when using a pretrained model, it's important that your custom data going into the model is prepared in the same way as the original training data that went into the model.

In [ ]:
# Get a set of pretrained model weights
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT  # .DEFAULT = best available weights from pretraining on ImageNet
weights

In [ ]:
# Get the transform used to create our pretrained weights
auto_transforms = weights.transforms()
auto_transforms

We can use `auto_transforms` to create DataLoaders with `create_dataloaders()` just as before.



In [ ]:
# Create training and testing DataLoader as well as get a list of class names
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                                test_dir=test_dir,
                                                                                transform=auto_transforms,
                                                                                batch_size=32)

train_dataloader, test_dataloader, class_names

### Getting a pretrained model
Over the past few notebooks we've been building PyTorch neural networks from scratch.

And while that's a good skill to have, our models haven't been performing as well as we'd like.

That's where transfer learning comes in.

## We use the Pretrained model
**Based on the problem/the device we working with**

In [ ]:
# OLD: Setup the model with pretrained weights and send it to the target device (this was prior to torchvision v0.13)
# model = torchvision.models.efficientnet_b0(pretrained=True).to(device) # OLD method (with pretrained=True)

# NEW: Setup the model with pretrained weights and send it to the target device (torchvision v0.13+)
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # .DEFAULT = best available weights
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

#model # uncomment to output (it's very long)

##Getting a summary of our torch model with `torchinfo.summary()`


In [ ]:
# Print a summary using  torchinfo (uncomment for actual output)
summary(model=model,
        input_size=(32, 3, 224, 224), # Make sure this is "input_size", not "input+_hape"
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
        )

##Freezing the base model and changing the output layer to suit our needs

The process of transfer learning usually goes: freeze some base layers of a pretrained model (typically the `features` section) and then adjust the output layers (also called head/classifier layers) to suit your needs.

In [ ]:
# Freeze all base layers in the "features" section of the model (the feature extractor) by setting requires_grad=False
for param in model.features.parameters():
    param.requires_grad = False


Our new classifier layer should be on the same device as our model.

In [ ]:
# set the manual_seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Get the length of class_names  (one output unit for eac class)
output_shape = len(class_names)

# Recreate the classifier layer and seed it to the target device
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=output_shape, # same number of units as the number of
                    bias=True)).to(device)

output layer is updated let's get another summary of our model and see the results


In [ ]:
# Do a summary after freezing the features and changing the output classifier
summary(model,
        input_size=(32, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

##Train Model
Now we've got a pretrained model that's semi-frozen and has a customised classifier, how about we see transfer learning in action?

To begin training, let's create a loss function and an optimizer.

Because we're still working with multi-class classification, we'll use nn.CrossEntropyLoss() for the loss function.

And we'll stick with torch.optim.Adam() as our optimizer with lr=0.001.

In [ ]:
# Define the loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


To train our model, we can use train() function we defined in the 05. PyTorch Going Modular section 04.

The train() function is in the engine.py script inside the going_modular directory.

Let's see how long it takes to train our model for 5 epochs.

>    Note: We're only going to be training the parameters classifier here as all of the other parameters in our model have been frozen.


In [ ]:
# Set the random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Setup the training and save the results
results = engine.train(model=model,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=5,
                       device=device)

# End the timer and print out how  long it took
end_time = timer()
print(f"[INFO] Toatl training time: {end_time-start_time: .3f} seconds")

## Evalutte model bt plotting the loss curves


In [ ]:
# Get the plot_loss_curvers() function from helper_functions.py , download the file if we don't have it
try:
  from helper_functions import plot_loss_curves
except:
  print(f"[INFO] Couldn't find helper_functions.py, downloading....")
  with open("helper_functions.py", "wb")as f:
    import requests
    request =   requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    f.write(request.content)
  from helper_functions import plot_loss_curves
# Plot the loss curves of our model
plot_loss_curves(results)

## Make predictions on images from the test set

To do all of this, we'll create a function pred_and_plot_image() to:

    Take in a trained model, a list of class names, a filepath to a target image, an image size, a transform and a target device.
    Open an image with PIL.Image.open().
    Create a transform for the image (this will default to the manual_transforms we created above or it could use a transform generated from weights.transforms()).
    Make sure the model is on the target device.
    Turn on model eval mode with model.eval() (this turns off layers like nn.Dropout(), so they aren't used for inference) and the inference mode context manager.
    Transform the target image with the transform made in step 3 and add an extra batch dimension with torch.unsqueeze(dim=0) so our input image has shape [batch_size, color_channels, height, width].
    Make a prediction on the image by passing it to the model ensuring it's on the target device.
    Convert the model's output logits to prediction probabilities with torch.softmax().
    Convert model's prediction probabilities to prediction labels with torch.argmax().
    Plot the image with matplotlib and set the title to the prediction label from step 9 and prediction probability from step 8.


In [ ]:
from typing import List, Tuple

from PIL import Image

# 1 take in a trained model ,class names , image path , image size , a transform and a target device
def pred_and_plot_image(model:torch.nn.Module,
                        image_path: str,
                        class_names: List[str],
                        image_size: Tuple[int, int] = (224, 224),
                        transform: torchvision.transforms = None,
                        device: torch.device=device):

  # 2 open Image
  img = Image.open(image_path)

  # 3 Create transformation for image (if one doesn't exist)
  if transform is not None:
    image_transform = transform
  else:
    image_transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    # Predict on image
    # 4 Make sure the model is on the traget device
    model.to(device)

    # 5 Turn on model evaluation mode and inference mode
    model.eval()
    with torch.inference_mode():
      # 6 Transform and add an extra dimension to image (model requires samplesn [batch_size, color_channels, height, width])
      transformed_image = image_transform(img).unsqueeze(dim=0)

      # 7. Make a prediction on image with an extra dimension and send it to the target device
      target_image_pred = model(transformed_image.to(device))

    #   8. Convert logits -> prediction probabilities (using torch.softmax() for multi-class classification)
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1)

    #  9 Convert predictions probabilities -> prediction labels
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

    # 10  Plot image with predicted label and probability
    plt.figure()
    plt.imshow(img)
    plt.title(f"Pred: {class_names[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}")
    plt.axis(False);






In [ ]:
# Get a random list of image paths from test set
import random
num_images_to_plot = 3
test_image_path_list = list(Path(test_dir).glob("*/*.jpg"))  # get list all image paths from test data
test_image_path_sample = random.sample(population=test_image_path_list, # go through all of the test image paths
                                       k=num_images_to_plot) # randomly select 'k' image paths to pred and plot
# Make predictions on and plot the images
for image_path in test_image_path_sample:
  pred_and_plot_image(model=model,
                      image_path=image_path,
                      class_names=class_names,

                        # transform=weights.transforms(), # optionally pass in a specified transform from our pretrained model weights
                      image_size=(224, 224)
                        )

## Making  Predictions on a custom image


In [ ]:
# Download the custom image
import requests

# Setup custom image path
custom_image_path = data_path / "04-pizza-dad.jpeg"

# Download the image if it doesb't exist
if not custom_image_path.is_file():
  with open(custom_image_path, "wb") as f:
    # When downloading from GitHub , need to use the "raw" file link
     request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
     print(f"Downloading {custom_image_path}...")
     f.write(request.content)

else:
  print(f"{custom_image_path} already exist, skipping the downlaod.........")


# Predict on custom image
pred_and_plot_image(model=model,
                    image_path=custom_image_path,
                    class_names=class_names)